In [73]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

In [3]:
df=pd.read_csv('/kaggle/input/datasets/brendan45774/test-file/tested.csv')

In [6]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [7]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,418.000000,418.000000,418.000000,332.000000,418.000000,418.000000,417.000000
mean,1100.500000,0.363636,2.265550,30.272590,0.447368,0.392344,35.627188
std,120.810458,0.481622,0.841838,14.181209,0.896760,0.981429,55.907576
min,892.000000,0.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,996.250000,0.000000,1.000000,21.000000,0.000000,0.000000,7.895800
50%,1100.500000,0.000000,3.000000,27.000000,0.000000,0.000000,14.454200
75%,1204.750000,1.000000,3.000000,39.000000,1.000000,0.000000,31.500000
max,1309.000000,1.000000,3.000000,76.000000,8.000000,9.000000,512.329200


In [10]:
df.isnull().sum()/len(df)*100

PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            20.574163
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.239234
Cabin          78.229665
Embarked        0.000000
dtype: float64

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     object 
 11  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 39.3+ KB


In [15]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [19]:
#workable data 
data=df.drop(['PassengerId','Name','Cabin'],axis=1)

In [33]:
data['Age'].mode()

0    21.0
1    24.0
Name: Age, dtype: float64

In [37]:
data['Age']=data['Age'].fillna(data['Age'].median())

In [57]:
data=data.dropna()

In [58]:
data.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Ticket      0
Fare        0
Embarked    0
dtype: int64

In [59]:
data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,0,3,0,34.5,0,0,330911,7.8292,Q
1,1,3,1,47.0,1,0,363272,7.0000,S
2,0,2,0,62.0,0,0,240276,9.6875,Q
3,0,3,0,27.0,0,0,315154,8.6625,S
4,1,3,1,22.0,1,1,3101298,12.2875,S


In [61]:
data.select_dtypes('object').info()

<class 'pandas.core.frame.DataFrame'>
Index: 417 entries, 0 to 417
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Ticket    417 non-null    object
 1   Embarked  417 non-null    object
dtypes: object(2)
memory usage: 9.8+ KB


In [50]:
data['Sex']=data['Sex'].map({'male':0,'female':1})

In [63]:
data=data.drop('Ticket',axis=1)

In [65]:
data=pd.get_dummies(data)

In [66]:
data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
0,0,3,0,34.5,0,0,7.8292,False,True,False
1,1,3,1,47.0,1,0,7.0000,False,False,True
2,0,2,0,62.0,0,0,9.6875,False,True,False
3,0,3,0,27.0,0,0,8.6625,False,False,True
4,1,3,1,22.0,1,1,12.2875,False,False,True


In [88]:
x=data.drop('Fare',axis=1)

In [89]:
y=data['Fare']

In [90]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.1,random_state=42)

In [91]:
ss=StandardScaler()

In [92]:
x_train=ss.fit_transform(x_train)

In [93]:
x_test=ss.transform(x_test)

In [118]:
#model
model=Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),
    Dropout(0.3),

    Dense(32,activation='relu'),
    Dropout(0.2),

    Dense(16,activation='relu'),
    Dropout(0.2),

    Dense(1,activation='linear')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [119]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 64)             │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,265 (12.75 KB)

 Trainable params: 3,265 (12.75 KB)

 Non-trainable params: 0 (0.00 B)

In [120]:
model.compile(optimizer='adam',loss='mse',metrics=['mae'])

In [121]:
early_stop=EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True
)

In [122]:
history=model.fit(x_train,y_train,epochs=300,batch_size=1,validation_split=0.1,callbacks=[early_stop])

Epoch 1/300
337/337 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 2728.5718 - mae: 26.4757 - val_loss: 313.0354 - val_mae: 12.5736
Epoch 2/300
337/337 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2207.0027 - mae: 28.2796 - val_loss: 581.3041 - val_mae: 14.3240
Epoch 3/300
337/337 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2750.2854 - mae: 27.2108 - val_loss: 469.2794 - val_mae: 12.5473
Epoch 4/300
337/337 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2120.8115 - mae: 23.8664 - val_loss: 353.2761 - val_mae: 11.0102
Epoch 5/300
337/337 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1849.7747 - mae: 21.8401 - val_loss: 410.3017 - val_mae: 11.6448
Epoch 6/300
337/337 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1855.5859 - mae: 22.5310 - val_loss: 496.8861 - val_mae: 12.3855
Epoch 7/300
337/337 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2048.3044 - mae: 24.4985 - val_loss: 419.8221 - val_mae: 11.9684
Epoch 8/300
337/337 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1454.1694 - mae: 19.9737 - val_loss: 1067.6216 - val_mae:

In [123]:
y_pred=model.predict(x_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step


In [129]:
r2=r2_score(y_test,y_pred)

In [138]:
r2

0.9723242420032